In [1]:
import io
import os
import pathlib
import subprocess

from IPython.display import display, Markdown
import polars

from utils import list_code

EVALUATION_DIR = pathlib.Path.cwd()
SCENARIO_DIR = EVALUATION_DIR / "scenarios"

os.environ["EVALUATION_DIR"] = str(EVALUATION_DIR)
os.environ["SCENARIO_DIR"] = str(SCENARIO_DIR)

# Scenarios to Create Traffic Data

In [2]:
with open(SCENARIO_DIR / "README.md") as f:
    display(Markdown(f.read()))

See [install instructions](https://docs.docker.com/engine/install/ubuntu/)

Requires at least

- Ubuntu 24.04
- Docker version 28.0.0
- Docker Compose version 2.33.1

```sh
for pkg in docker.io docker-doc docker-compose docker-compose-v2 podman-docker containerd runc; do sudo apt-get autoremove $pkg; done

# Add Docker's official GPG key:
sudo apt-get update
sudo apt-get install ca-certificates curl
sudo install -m 0755 -d /etc/apt/keyrings
sudo curl -fsSL https://download.docker.com/linux/ubuntu/gpg -o /etc/apt/keyrings/docker.asc
sudo chmod a+r /etc/apt/keyrings/docker.asc

# Add the repository to Apt sources:
echo \
  "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.asc] https://download.docker.com/linux/ubuntu \
  $(. /etc/os-release && echo "${UBUNTU_CODENAME:-$VERSION_CODENAME}") stable" | \
  sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
sudo apt-get update

sudo apt-get install docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin
sudo usermod -a -G docker "${USER}"
```


In [3]:
list_code(SCENARIO_DIR / "run.sh")

#! /bin/bash
#
# run.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR=$( cd -- "$( dirname -- "$(realpath "$0")" )" &> /dev/null && pwd )

export HOST_GID=$(id -g)
export HOST_UID=$(id -u)

chmod -R o+r ${SCRIPT_DIR}/database/ ${SCRIPT_DIR}/../jsons/
chmod o+x ${SCRIPT_DIR}/database/ ${SCRIPT_DIR}/../jsons/

if ! ls ${SCRIPT_DIR}/../jsons/*.sqlite3 &> /dev/null; then
    echo "No base database found!" >&2
    exit 1
fi

MAIN_ENV="${SCRIPT_DIR}"/.env
DATA_ENVS=(
    "${SCRIPT_DIR}"/.json.env
    "${SCRIPT_DIR}"/.cbor.env
)
DNS_ENVS=(
    "${SCRIPT_DIR}"/.dns-msg.env
    "${SCRIPT_DIR}"/.dns-cbor.env
)
SECURITIES=(
    "transport"
    "object"
    "object-base"
    ""
)
LINK_LAYERS=(
    ""
    # "schc" 
)
PROTOCOLS=(
    "coap"
    # "https"
)
NETWORK_SETUPS=("d1" "d2" "p1" "p2")
BLOCKWISE=(
    ""
    # "block"
)

if id | grep -q "=0(root)" || id | grep -vq "docker"; then
    echo "Executing user must not be root and must be in the 'docker' group" >&2
    exit 1
fi

DOCKER_COMPOSE_PIDS=()

kill_docker() {
    for pid in ${DOCKER_COMPOSE_PIDS[@]}; do
        kill -SIGTERM "${pid}"
        tail --pid="${pid}" -f /dev/null
    done
    reset
    exit
}

trap kill_docker SIGHUP SIGTERM SIGINT SIGQUIT SIGABRT

if [ "$1" = "--build" ] || ! docker image ls | grep -q "pivot-eval/"; then
    docker system prune -f

    for l2 in "${LINK_LAYERS[@]}"; do
        PREFIX_HINT_1=6
        for prot in "${PROTOCOLS[@]}"; do
            PREFIX_HINT_2=0
            for setup in "${NETWORK_SETUPS[@]}"; do
                setup_iface=$(echo "${setup}" | sed -E -e 's/([dp])1/\1i/g' -e 's/([dp])2/\1ii/g')
                l2_iface=$(echo "${l2}" | sed -E -e 's/-//g' -e 's/schc/0/g')
                export DATABASE_BACKEND_PREFIX="fd00:${PREFIX_HINT_1}b${PREFIX_HINT_2}6::"
                export WPAN_SIMULATION_NAME="${prot}${l2}-${setup}-wpan-simulation"
                export WPAN_SIMULATION_IFACE="${prot}${l2_iface}${setup}_wpan"
                export WPAN_SIMULATION_PREFIX="fdd8:${PREFIX_HINT_1}b${PREFIX_HINT_2}6:eccc::"
                export UPSTREAM_NAME="${prot}${l2}-${setup}-upstream"
                export UPSTREAM_IFACE="${prot}${l2_iface}${setup}_upstream"
                export UPSTREAM_PREFIX="fdd8:${PREFIX_HINT_1}b${PREFIX_HINT_2}6:ecc0::"
                COMPOSE_BAKE=true DATA_FORMAT=application/cbor DNS_FORMAT=application/dns+cbor \
                    docker compose --env-file "${MAIN_ENV}" \
                        ${ADDITIONAL_OPTS} \
                        -f "${SCRIPT_DIR}/docker-compose-${prot}${l2}-${setup}.yaml" build
                PREFIX_HINT_2=$(( PREFIX_HINT_2 + 1 ))
            done
            PREFIX_HINT_1=$(( PREFIX_HINT_1 + 1 ))
        done
    done

    docker image prune -f
fi 

for data_env in "${DATA_ENVS[@]}"; do
    for dns_env in "${DNS_ENVS[@]}"; do
        for sec in "${SECURITIES[@]}"; do
            for l2 in "${LINK_LAYERS[@]}"; do
                PREFIX_HINT_1=6
                for prot in "${PROTOCOLS[@]}"; do
                    if [ "$prot" != "coap" -a "$sec" != "transport" ]; then
                        continue
                    fi
                    for block in "${BLOCKWISE}"; do
                        ALL_SUCCESSFUL=0
                        while [ ${ALL_SUCCESSFUL} -eq 0 ]; do
                            PREFIX_HINT_2=0
                            unset DOCKER_COMPOSE_PIDS
                            for setup in "${NETWORK_SETUPS[@]}"; do
                                setup_iface=$(echo "${setup}" | sed -E -e 's/([dp])1/\1i/g' -e 's/([dp])2/\1ii/g')
                                l2_iface=$(echo "${l2}" | sed -E -e 's/-//g' -e 's/schc/0/g')
                                export DATABASE_BACKEND_PREFIX="fd00:${PREFIX_HINT_1}b${PREFIX_HINT_2}6::"
                                export WPAN_SIMULATION_NAME="${prot}${l2}-${setup}-wpan-simulation"
                                export WPAN_SIMULATION_IFACE="${prot}${l2_iface}${setup}_wp

In [4]:
%%bash

tmux new-session -s "scenarios" -d "${SCENARIO_DIR}/run.sh"

## Check Logs

In [5]:
list_code(SCENARIO_DIR / "count_client_logs.sh")

#! /bin/bash
#
# count_client_logs.sh
# Copyright (C) 2025 TU Dresden
#
# Distributed under terms of the MIT license.
#

SCRIPT_DIR=$( cd -- "$( dirname -- "$(realpath "$0")" )" &> /dev/null && pwd )
OUTPUT_DATASETS="$(readlink -f "${OUTPUT_DATASETS:-${SCRIPT_DIR}/../output_dataset}")"

awk '
    {
        timestamps[FILENAME][$1 * 10**9] += 1
    }
    END {
        for (fn in timestamps) {
            sum = 0; min=0;
            for (ts in timestamps[fn]) {
                if ((!min || min > ts) && ts != 0) { min = ts }
                sum += timestamps[fn][ts]
            }
            print min, fn, length(timestamps[fn]), sum
        }
    }' "${OUTPUT_DATASETS}"/*.client.log | \
        sort -n | awk '{print $3, $4, $2}'

In [6]:
EXPECTED_REQUESTS = 120699

proc = subprocess.check_output(f"{SCENARIO_DIR}/count_client_logs.sh", text=True)
df = polars.read_csv(
    io.StringIO(proc),
    has_header=False,
    separator=" ",
    new_columns=["requests", "lines", "filename"]
)
df = df.with_columns(
    df["filename"].map_elements(
        lambda path: str(pathlib.Path(path).relative_to(EVALUATION_DIR)),
        return_dtype=str,
    )
)
with polars.Config(
    tbl_rows=df.shape[0],
    fmt_str_lengths=df["filename"].str.len_chars().max()
):
    display(df)
if (df["requests"] == EXPECTED_REQUESTS).all():
    display(Markdown(f"All {df.shape[0]} files contain the expected {EXPECTED_REQUESTS} requests"))
else:
    display(Markdown(f"The following logs are **incomplete**:"))
    display(df).filter(df["requests"] != EXPECTED_REQUESTS)

requests,lines,filename
i64,i64,str
120699,123347,"""output_dataset/coaps-d1_json_dns_message.client.log"""
120699,123347,"""output_dataset/coaps-d2_json_dns_message.client.log"""
120699,123347,"""output_dataset/coaps-p1_json_dns_message.client.log"""
120699,123347,"""output_dataset/coaps-p2_json_dns_message.client.log"""
120699,123732,"""output_dataset/oscore-d1_json_dns_message.client.log"""
120699,123732,"""output_dataset/oscore-d2_json_dns_message.client.log"""
120699,127302,"""output_dataset/oscore-p1_json_dns_message.client.log"""
120699,127302,"""output_dataset/oscore-p2_json_dns_message.client.log"""
120699,123347,"""output_dataset/coap-d1_json_dns_message.client.log"""


All 48 files contain the expected 120699 requests